# Data Validation - Telco Customer Churn

This notebook checks the dataset before EDA and modeling.

- **Dataset:** IBM Telco Customer Churn
- **Target column:** Churn

## Load Data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/Telco_Customer_Churn.csv")
df.head()

## 1. Dataset Shape Validation

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

if df.shape[0] > 0 and df.shape[1] > 0:
    print("Dataset is not empty")
else:
    print("Dataset is empty")

**Note:** We have data to analyze. Empty file would be a problem.

## 2. Schema Validation

In [ ]:
print(df.columns.tolist())
print("\nNumber of columns:", len(df.columns))
print("Duplicate columns:", df.columns.duplicated().sum())
print("Churn column exists:", "Churn" in df.columns)

**Note:** Churn column must be there because it is our target.

## 3. Data Type Validation

In [ ]:
df.dtypes

In [ ]:
df.info()

In [ ]:
# TotalCharges is object - convert for checks
print("TotalCharges type:", df["TotalCharges"].dtype)
df["TotalCharges_num"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

**Note:** TotalCharges should be converted to number in data cleaning step.

## 4. Missing Value Validation

In [ ]:
df.isnull().sum()

In [ ]:
(df.isnull().sum() / len(df) * 100).round(2)

In [ ]:
blank_total = (df["TotalCharges"].astype(str).str.strip() == "").sum()
print("Blank TotalCharges:", blank_total)
print("Missing after numeric conversion:", df["TotalCharges_num"].isnull().sum())

**Note:** Some new customers have blank TotalCharges when tenure is 0.

## 5. Duplicate Record Validation

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate customerID:", df["customerID"].duplicated().sum())

**Note:** No duplicates is good - each row is one customer.

## 6. Unique Value Analysis

In [ ]:
df.nunique()

In [ ]:
print("Unique customerID:", df["customerID"].nunique())
print("Total rows:", len(df))

**Note:** customerID is unique for each row. Other columns like gender have only 2 values.

## 7. Numerical Data Validation

In [ ]:
df[["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges_num"]].describe()

In [ ]:
print("Negative tenure:", (df["tenure"] < 0).sum())
print("Negative MonthlyCharges:", (df["MonthlyCharges"] < 0).sum())
print("Negative TotalCharges:", (df["TotalCharges_num"] < 0).sum())

**Note:** Values look reasonable. Tenure is between 0 and 72 months.

## 8. Categorical Data Validation

In [ ]:
cols = ["gender", "Partner", "InternetService", "Contract", "PaymentMethod", "Churn"]

for col in cols:
    print(col, ":", df[col].unique())

**Note:** Categories look normal - Yes/No, contract types, payment methods etc.

## 9. Target Variable Validation

In [ ]:
df["Churn"].value_counts()

In [ ]:
churn_rate = (df["Churn"] == "Yes").mean() * 100
print("Churn %:", round(churn_rate, 2))

**Note:** About 27% churn. More customers stay than leave (imbalanced data).

## 10. Data Leakage Check

In [ ]:
print("Do not use customerID in model - it is just an ID")
print("Do not use Churn as feature - it is the target")
print("TotalCharges can be used but it depends on tenure")
print("Other columns are fine to use as features")

**Note:** Using target as feature would cheat the model. ID has no meaning for prediction.

## 11. Business Rule Validation

In [ ]:
print("Tenure = 0 customers:", (df["tenure"] == 0).sum())

tenure_zero = df[df["tenure"] == 0]
blank = (tenure_zero["TotalCharges"].astype(str).str.strip() == "").sum()
print("Blank TotalCharges when tenure is 0:", blank)

In [ ]:
mask = (df["tenure"] >= 1) & df["TotalCharges_num"].notna()
issue = df[mask & (df["TotalCharges_num"] < df["MonthlyCharges"])]
print("TotalCharges less than MonthlyCharges:", len(issue))

**Note:** New customers (tenure 0) may not have total charges yet. That is normal.

## 12. Validation Summary

In [ ]:
print("SUMMARY")
print("--------")
print("Rows:", df.shape[0], "| Columns:", df.shape[1])
print("Target column OK: Churn present")
print("Duplicates: none")
print("Issue: TotalCharges needs numeric conversion")
print("Issue: 11 missing TotalCharges (tenure 0)")
print("Churn rate: ~27%")
print()
print("Before EDA:")
print("1. Fix TotalCharges datatype")
print("2. Handle missing TotalCharges")
print("3. Remove customerID from features")
print("4. Start EDA notebook")